In [6]:
!git clone https://github.com/VinAIResearch/COVID19Tweet.git
%cd COVID19Tweet
!unzip -q WNUT-2020-Task-2-Dataset.zip -d data
!find data -type f

Cloning into 'COVID19Tweet'...
remote: Enumerating objects: 47, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 47 (delta 14), reused 26 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (47/47), 6.42 MiB | 13.90 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/COVID19Tweet/COVID19Tweet
data/WNUT-2020-Task-2-Dataset/train.tsv
data/WNUT-2020-Task-2-Dataset/test.tsv
data/WNUT-2020-Task-2-Dataset/unlabeled_test_with_noise.tsv
data/WNUT-2020-Task-2-Dataset/valid.tsv


In [7]:
# 1. 함수 가져오기
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer #문자를 숫자로 가져옴
from sklearn.linear_model import LogisticRegression #로지스틱회귀 사용
from sklearn.metrics import accuracy_score, f1_score, classification_report #정확도, 점수, 성능표 보기

In [8]:
# 2. 데이터 읽기
train_df = pd.read_csv("data/WNUT-2020-Task-2-Dataset/train.tsv", sep="\t") #탭으로 나눠져있음
valid_df = pd.read_csv("data/WNUT-2020-Task-2-Dataset/valid.tsv", sep="\t", header=None)
valid_df.columns = ['Id', 'Text', 'Label'] #헤더가 없다고 함 + 열 이름 할당하게 되어 이름도 다시 할당

In [9]:
# 3. 데이터 확인
train_df.head()
train_df.columns
train_df.shape

(6936, 3)

In [10]:
# 4. 입력과 정답 나누기

X_train = train_df["Text"].astype(str)
y_train = train_df["Label"].astype(str)

X_valid = valid_df["Text"].astype(str)
y_valid = valid_df["Label"].astype(str)

In [11]:
# 5. 텍스트 숫자화
vectorizer = TfidfVectorizer()

X_train_vec = vectorizer.fit_transform(X_train) #단어사전 만든 후 숫자로 바꿈
X_valid_vec = vectorizer.transform(X_valid) #train에서 만든 기준으로 숫자로만 바꾼다

In [12]:
# 6. 모델 학습
model = LogisticRegression(max_iter=1000) #로지스틱 회귀 모델 만들기
model.fit(X_train_vec, y_train) #학습 시작

LogisticRegression(max_iter=1000)

In [13]:
# 7. 예측
pred = model.predict(X_valid_vec)

In [14]:
# 8. 성능 확인
print(accuracy_score(y_valid, pred))
print(f1_score(y_valid, pred, pos_label="INFORMATIVE"))
print(classification_report(y_valid, pred))

0.817
0.7941507311586051
               precision    recall  f1-score   support

  INFORMATIVE       0.85      0.75      0.79       472
UNINFORMATIVE       0.80      0.88      0.84       528

     accuracy                           0.82      1000
    macro avg       0.82      0.81      0.81      1000
 weighted avg       0.82      0.82      0.82      1000



In [15]:
# 9. 예측 결과를 valid_df에 붙이기 (copy로 리스트 복사)
result_df = valid_df.copy()
result_df["Predicted_Label"] = pred

# 10. 모델이 INFORMATIVE라고 예측한 문장만 가져오기 (선별)
informative_df = result_df[result_df["Predicted_Label"] == "INFORMATIVE"].copy()

In [16]:
# 11. information 문장 중 타겟 정보 분류하기

category_keywords = {
    "확진자 정보": r"\b(confirmed|positive)\b",
    "의심 사례/증상자 정보": r"\b(suspected|symptoms|symtoms|under observation)\b",
    "사망자 정보": r"\b(death|died|fatality|deceased)\b",
    "회복/퇴원 정보": r"\b(recovered|discharged)\b",
    "검사 정보": r"\b(tested|tests|screened)\b"
}

for category, pattern in category_keywords.items():
    informative_df[category] = informative_df["Text"].str.contains(
        pattern,
        case=False, #대소문자 무시
        na=False,
        regex=True
    )

/tmp/ipykernel_8773/3626487242.py:12: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  informative_df[category] = informative_df["Text"].str.contains(
/tmp/ipykernel_8773/3626487242.py:12: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  informative_df[category] = informative_df["Text"].str.contains(
/tmp/ipykernel_8773/3626487242.py:12: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  informative_df[category] = informative_df["Text"].str.contains(
/tmp/ipykernel_8773/3626487242.py:12: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  informative_df[category] = informative_df["Text"].str.contains(
/tmp/ipykernel_8773/3626487242.py:12: UserWarning: T

In [17]:
# 12. 해당되는 카테고리 이름을 하나의 컬럼으로 정리

def get_categories(row):
    matched = []
    for category in category_keywords.keys():
        if row[category]:
            matched.append(category)

    if matched:
        return ", ".join(matched)
    else:
        return "기타 information"

informative_df["Target_Category"] = informative_df.apply(get_categories, axis=1)

In [18]:
# 13. 타겟 키워드가 하나라도 포함된 information 문장만 보기

target_df = informative_df[informative_df["Target_Category"] != "기타 information"]

target_df[["Text", "Target_Category"]].head(20)

,Text,Target_Category
4,Report suggested that the actual number of und...,확진자 정보
9,UPDATE: the two people at our Hail Creek opera...,검사 정보
17,"The Taslee Palm City Estate in Maitama, Abuja,...",확진자 정보
18,Soooo.... more than 1000 people died above nor...,사망자 정보
21,There are eight new cases of #COVID19 in #Sask...,검사 정보
22,"240,000 Americans died from coronavirus and, w...",사망자 정보
25,NEWS: A permanent memorial is going to be set ...,사망자 정보
26,#KatieMiller is #StephenMillers new wife &amp;...,확진자 정보
27,i didn’t know I’ll wake up from the news of de...,확진자 정보
39,"The 19th PH death is PH205, a 73-year-old Fili...","확진자 정보, 사망자 정보, 검사 정보"


In [19]:
confirmed_df = informative_df[informative_df["확진자 정보"] == True] #특정 정보만 보고싶을때의 코드

confirmed_df[["Text", "Target_Category"]].head(20)

,Text,Target_Category
4,Report suggested that the actual number of und...,확진자 정보
17,"The Taslee Palm City Estate in Maitama, Abuja,...",확진자 정보
26,#KatieMiller is #StephenMillers new wife &amp;...,확진자 정보
27,i didn’t know I’ll wake up from the news of de...,확진자 정보
39,"The 19th PH death is PH205, a 73-year-old Fili...","확진자 정보, 사망자 정보, 검사 정보"
44,@USER @USER @USER A person who visited the nur...,확진자 정보
45,@USER COVID-19 deaths in the US have exceeded ...,확진자 정보
48,Ouch. 2 #coronavirus cases in Broward County (...,"확진자 정보, 사망자 정보, 검사 정보"
64,Cherokee County residents concerned after posi...,확진자 정보
69,@USER @USER Sure been taking them out of mine....,"확진자 정보, 검사 정보"
